# Dynex SDK - nBit Adder Native Gate Circuit Example

First we import the required packages:

In [6]:
from pennylane import numpy as np
import pennylane as qml
from dynex import DynexConfig, ComputeBackend, DynexCircuit

config = DynexConfig(compute_backend=ComputeBackend.QPU, qpu_model="apollo_rc1", use_notebook_output=True)

We define our circuit:

In [7]:
params = [28938284928182, 233312722366482869645212131]  # two numbers to add


def Nqubits(a, b):
    mxVal = a + b
    return mxVal.bit_length()


wires = Nqubits(*params)


def Kfourier(k, wires):
    for j in range(len(wires)):
        qml.RZ(k * np.pi / (2 ** j), wires=wires[j])


def FullAdder(params, state=True):
    a, b = params
    wires = Nqubits(a, b)
    qml.BasisEmbedding(a, wires=range(wires))
    qml.QFT(wires=range(wires))
    Kfourier(b, range(wires))
    qml.adjoint(qml.QFT)(wires=range(wires))
    if state:
        return qml.state()
    else:
        return qml.sample()

We draw the circuit:

In [8]:
# draw circuit:
# too large to draw _ = qml.draw_mpl(FullAdder, style="black_white")(params)
print("Cant't draw circuit with", Nqubits(params[0], params[1]), "wires");

Cant't draw circuit with 88 wires


We execute and measure the circuit on the Dynex platform:

In [9]:
# Execute the circuit on Dynex:
dynex_circuit = DynexCircuit(config=config)
measure = dynex_circuit.execute(FullAdder, params, wires, method="measure",
                                num_reads=10, integration_steps=10)
print("Mesaure:", measure)

INFO: [DYNEX-APOLLO-RC1] Executing PennyLane quantum circuit
INFO: [DYNEX-APOLLO-RC1] Sampler initialised
INFO: [DYNEX-APOLLO-RC1] Apollo QPU chip: apollo_rc1
INFO: [DYNEX-APOLLO-RC1] Settings: num_reads=10, shots=1, annealing_time=10
INFO: [DYNEX-APOLLO-RC1] Submitting the job to Dynex.
INFO: [DYNEX-APOLLO-RC1] SUCCESS: Job created successfully (job_id=7417)
INFO: [DYNEX-APOLLO-RC1] feed_dict: {'cos_rz_0': 0.7730741440608246, 'cos_rz_1': -0.9415609762678211, 'cos_rz_10': 0.10478898165242552, 'cos_rz_11': 0.7432324608264986, 'cos_rz_12': -0.9336038937436204, 'cos_rz_13': -0.18220332908097422, 'cos_rz_14': 0.6394515896137196, 'cos_rz_15': -0.9053870966646586, 'cos_rz_16': 0.21750046360334663, 'cos_rz_17': -0.7802244752644416, 'cos_rz_18': -0.33149323125484653, 'cos_rz_19': 0.5781465077059419, 'cos_rz_2': -0.17093715765183842, 'cos_rz_20': 0.8882979533090071, 'cos_rz_21': 0.9716732869923427, 'cos_rz_22': -0.9928930675033295, 'cos_rz_23': 0.05961095745192508, 'cos_rz_24': 0.72787737890798

Mesaure: [1 1 0 0 0 0 0 0 1 1 1 1 1 1 0 1 1 1 1 0 0 1 0 0 0 0 0 0 0 1 0 1 1 1 0 1 1
 1 0 1 1 0 1 1 0 0 0 0 1 0 1 0 1 0 0 1 0 0 0 1 1 0 0 1 1 0 1 1 0 0 0 1 1 0
 0 0 0 1 1 0 1 0 0 1 1 0 0 1]


In [10]:
bitStr = "".join(map(str, measure.astype(int)))
dynexResult = int(bitStr, 2)
print("Dynex Result:", dynexResult)
print("Expected Result:", sum(params))
isValidDynex = dynexResult == sum(params)
print("Is Dynex Result Valid?", isValidDynex)

Dynex Result: 233312722366511807930140313
Expected Result: 233312722366511807930140313
Is Dynex Result Valid? True
